In [4]:
import re
import json
import time
import math
import random
import hashlib
from pathlib import Path
from typing import List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# -----------------------------
# 설정
# -----------------------------
MODEL_NAME = "gpt-4.1-mini"

INPUT_FILES = {
    "tires": "pirelli_f1_tires.txt",
    "circuits": "f1_circuits.json",
    "flags": "f1_flags_wiki.json",
    "intro": "f1_intro_wiki.json",
}

OUTPUT_DIR = Path("output_qa_pipeline_4docs")
OUTPUT_DIR.mkdir(exist_ok=True)

DOCS_JSON_PATH = OUTPUT_DIR / "normalized_docs.json"
CHUNKS_JSON_PATH = OUTPUT_DIR / "chunks.json"
RAW_QA_PATH = OUTPUT_DIR / "raw_qa.json"
DEDUP_QA_PATH = OUTPUT_DIR / "dedup_qa.json"
FINETUNE_JSONL_PATH = OUTPUT_DIR / "f1_finetune_qa.jsonl"

TEST_MODE = False
TEST_DOC_LIMIT = 20
TARGET_FINAL_QA = 1500

GENERATION_CONFIG = {
    "concept": 1,
    "summary": 1,
    "fact": 3,
    "comparison": 0,
    "misconception": 1,
}

SLEEP_SECONDS = 0.2
MAX_RETRIES = 3
MAX_PASSES = 10

CHUNK_SIZE = 1400
CHUNK_OVERLAP = 150

client = OpenAI()


# -----------------------------
# 1) 기본 유틸
# -----------------------------
def read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def read_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path: Path, data: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def normalize_text(text: str) -> str:
    text = text.replace("\u00ad", "")
    text = text.replace("\xa0", " ")
    text = text.replace("−", "-")
    text = text.replace("–", "-")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w가-힣\s-]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text[:80]


# -----------------------------
# 2) 문서 로드 및 통일
# -----------------------------
def load_tire_doc(path: str) -> List[Dict[str, Any]]:
    text = normalize_text(read_text(path))

    return [{
        "doc_id": "pirelli_f1_tires",
        "source_type": "text",
        "source_file": path,
        "category": "tires",
        "title": "Pirelli F1 Tires",
        "content": text,
        "metadata": {}
    }]


def load_circuit_docs(path: str) -> List[Dict[str, Any]]:
    rows = read_json(path)
    docs = []

    for row in rows:
        docs.append({
            "doc_id": row.get("doc_id", slugify(row.get("page_title", "circuit"))),
            "source_type": "json",
            "source_file": path,
            "category": row.get("category", "circuit"),
            "title": row.get("page_title", row.get("doc_id", "Circuit")),
            "content": normalize_text(row.get("content", "")),
            "metadata": row.get("metadata", {})
        })

    return docs


def load_wiki_docs(path: str, category_name: str) -> List[Dict[str, Any]]:
    rows = read_json(path)
    docs = []

    for idx, row in enumerate(rows, start=1):
        title = row.get("title", f"{category_name}_{idx}")
        content = row.get("content", "").strip()

        docs.append({
            "doc_id": f"{category_name}_{slugify(title)}_{idx}",
            "source_type": "json",
            "source_file": path,
            "category": category_name,
            "title": title,
            "content": normalize_text(content),
            "metadata": {
                "page_title": row.get("page_title", ""),
                "url": row.get("url", ""),
                "source": row.get("source", "")
            }
        })

    return docs


def load_all_docs() -> List[Dict[str, Any]]:
    docs = []
    docs.extend(load_tire_doc(INPUT_FILES["tires"]))
    docs.extend(load_circuit_docs(INPUT_FILES["circuits"]))
    docs.extend(load_wiki_docs(INPUT_FILES["flags"], "flags"))
    docs.extend(load_wiki_docs(INPUT_FILES["intro"], "intro"))
    return docs


# -----------------------------
# 3) 청킹
# -----------------------------
def split_long_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    text = normalize_text(text)

    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end].strip()

        # 너무 잘리는 느낌 줄이기 위해 문단/문장 경계 보정
        if end < text_len:
            last_break = max(
                chunk.rfind("\n\n"),
                chunk.rfind(". "),
                chunk.rfind("? "),
                chunk.rfind("! "),
            )
            if last_break > int(chunk_size * 0.6):
                chunk = chunk[:last_break + 1].strip()
                end = start + len(chunk)

        if chunk:
            chunks.append(chunk)

        if end >= text_len:
            break

        start = max(end - overlap, start + 1)

    return chunks


def make_chunks(docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    chunks = []

    for doc in docs:
        parts = split_long_text(doc["content"])

        for idx, part in enumerate(parts, start=1):
            chunks.append({
                "chunk_id": f"{doc['doc_id']}__chunk_{idx}",
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "category": doc["category"],
                "source_file": doc["source_file"],
                "content": part,
                "metadata": doc["metadata"],
                "chunk_index": idx,
                "num_chunks_in_doc": len(parts)
            })

    return chunks


# -----------------------------
# 4) 프롬프트
# -----------------------------
def build_prompt(chunk: Dict[str, Any], generation_config: Dict[str, int], pass_index: int = 1) -> str:
    question_type_desc = {
        "concept": "개념 설명형",
        "summary": "요약형",
        "fact": "사실 확인형",
        "comparison": "비교형",
        "misconception": "오해 교정형"
    }

    type_lines = []
    for qtype, n in generation_config.items():
        if n > 0:
            type_lines.append(f"- {qtype}: {n}개 ({question_type_desc[qtype]})")

    joined_types = "\n".join(type_lines)
    seed_hint = f"{chunk['chunk_id']}-{pass_index}-{random.randint(1000, 9999)}"

    prompt = f"""
너는 F1 입문자용 설명 챗봇의 파인튜닝 데이터를 만드는 역할이야.

아래 문서 일부를 읽고, 문서 내용만을 바탕으로 한국어 Q/A를 생성해라.

[문서 정보]
- 문서 ID: {chunk['doc_id']}
- 청크 ID: {chunk['chunk_id']}
- 제목: {chunk['title']}
- 카테고리: {chunk['category']}
- 회차: {pass_index}
- variation_hint: {seed_hint}

[문서 내용]
{chunk['content']}

[생성 개수]
{joined_types}

[생성 규칙]
- 문서 내용만 바탕으로 작성
- 한국어로 작성
- 초보자도 이해하기 쉽게 설명
- 답변은 2~4문장
- 숫자, 연도, 위치, 서킷명, 타이어명, 플래그명은 문서 기준 그대로 유지
- 문서에 없는 사실, 이유, 배경, 목적을 추론하지 말 것
- 질문끼리 의미 중복이 크지 않게 작성
- 오해 교정형은 왜 틀렸는지도 문서 범위 안에서 설명할 것
- 비교형은 같은 문서 청크 안에서 비교 가능한 대상이 있을 때만 만들 것
- 단순 문장 재진술만 하지 말고, 입문자 질문처럼 자연스럽게 만들 것
- 문서 제목만 반복하지 말고, 내용을 풀어서 설명할 것

- 아래 유형을 고루 섞기:
1. 개념형
2. 요약형
3. 사실 확인형
4. 비교형
5. 오해 교정형

[출력 형식]
반드시 아래 JSON 배열만 출력:
[
  {{
    "question": "...",
    "answer": "..."
  }}
]
""".strip()

    return prompt


def extract_json_array(text: str) -> str:
    match = re.search(r"\[\s*{.*?}\s*\]", text, re.DOTALL)
    if not match:
        raise ValueError("JSON 배열을 찾지 못했습니다.")
    return match.group(0)


def validate_qa_items(data: Any, chunk: Dict[str, Any]) -> List[Dict[str, Any]]:
    validated = []

    if not isinstance(data, list):
        return validated

    for item in data:
        if not isinstance(item, dict):
            continue

        question = item.get("question", "")
        answer = item.get("answer", "")

        if not isinstance(question, str) or not isinstance(answer, str):
            continue

        question = question.strip()
        answer = answer.strip()

        if not question or not answer:
            continue

        validated.append({
            "question": question,
            "answer": answer,
            "doc_id": chunk["doc_id"],
            "chunk_id": chunk["chunk_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "source_file": chunk["source_file"]
        })

    return validated


def generate_qa_for_chunk(chunk: Dict[str, Any], generation_config: Dict[str, int], pass_index: int = 1) -> List[Dict[str, Any]]:
    prompt = build_prompt(chunk, generation_config, pass_index)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=prompt
            )
            text = response.output_text.strip()

            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                cleaned = extract_json_array(text)
                data = json.loads(cleaned)

            validated = validate_qa_items(data, chunk)
            if validated:
                return validated

        except Exception as e:
            print(f"[재시도 {attempt+1}/{MAX_RETRIES}] {chunk['chunk_id']} 생성 실패: {e}")
            time.sleep(1.5)

    return []


# -----------------------------
# 5) 중복 제거
# -----------------------------
def normalize_question(q: str) -> str:
    q = q.strip().lower()
    q = re.sub(r"[^\w가-힣\s]", "", q)
    q = re.sub(r"\s+", " ", q)
    return q


def normalize_answer(a: str) -> str:
    a = a.strip().lower()
    a = re.sub(r"\s+", " ", a)
    return a


def make_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def deduplicate_qa(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    deduped = []

    for item in data:
        q_norm = normalize_question(item["question"])
        a_norm = normalize_answer(item["answer"])
        key = make_hash(q_norm + "||" + a_norm)

        if key in seen:
            continue

        seen.add(key)
        deduped.append(item)

    return deduped


# -----------------------------
# 6) 파인튜닝 형식 변환
# -----------------------------
def to_finetune_messages(item: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "messages": [
            {
                "role": "user",
                "content": item["question"]
            },
            {
                "role": "assistant",
                "content": item["answer"]
            }
        ]
    }


# -----------------------------
# 7) 목표 개수까지 생성
# -----------------------------
def generate_until_target(
    chunks: List[Dict[str, Any]],
    generation_config: Dict[str, int],
    target_final_qa: int
) -> List[Dict[str, Any]]:
    raw_qa: List[Dict[str, Any]] = []
    pass_index = 1

    while pass_index <= MAX_PASSES:
        print(f"\n===== PASS {pass_index}/{MAX_PASSES} 시작 =====")
        random.shuffle(chunks)

        for idx, chunk in enumerate(chunks, start=1):
            deduped_so_far = deduplicate_qa(raw_qa)

            if len(deduped_so_far) >= target_final_qa:
                print("목표 개수 도달. 생성 종료.")
                return raw_qa

            print(
                f"  - PASS {pass_index} | ({idx}/{len(chunks)}) "
                f"{chunk['chunk_id']} | {chunk['title']}"
            )

            qa_items = generate_qa_for_chunk(chunk, generation_config, pass_index=pass_index)
            raw_qa.extend(qa_items)

            if idx % 10 == 0:
                save_json(RAW_QA_PATH, raw_qa)
                dedup_snapshot = deduplicate_qa(raw_qa)
                save_json(DEDUP_QA_PATH, dedup_snapshot)
                print(
                    f"    누적 원본: {len(raw_qa)} | "
                    f"누적 중복제거: {len(dedup_snapshot)} / 목표 {target_final_qa}"
                )

            time.sleep(SLEEP_SECONDS)

        pass_index += 1

    print("최대 PASS 수에 도달했습니다.")
    return raw_qa


# -----------------------------
# 8) 실행
# -----------------------------
def main():
    print("[1/6] 문서 로드")
    docs = load_all_docs()
    print(f"전체 문서 수: {len(docs)}")

    if TEST_MODE:
        docs = docs[:TEST_DOC_LIMIT]
        print(f"테스트 문서 수: {len(docs)}")

    save_json(DOCS_JSON_PATH, docs)

    print("[2/6] 문서 청킹")
    chunks = make_chunks(docs)
    print(f"전체 청크 수: {len(chunks)}")
    save_json(CHUNKS_JSON_PATH, chunks)

    print("\n[청크 샘플 20개]")
    for c in chunks[:20]:
        print(f"{c['chunk_id']} | {c['category']} | {c['title']}")

    print("[3/6] 생성 설정 확인")
    per_chunk_raw = sum(GENERATION_CONFIG.values())
    print(f"청크당 요청 생성 수: {per_chunk_raw}")
    print(f"목표 최종 Q/A 수: {TARGET_FINAL_QA}")

    print("[4/6] Q/A 생성")
    raw_qa = generate_until_target(
        chunks=chunks,
        generation_config=GENERATION_CONFIG,
        target_final_qa=TARGET_FINAL_QA
    )
    save_json(RAW_QA_PATH, raw_qa)
    print(f"생성된 원본 Q/A 수: {len(raw_qa)}")

    print("[5/6] 중복 제거")
    deduped = deduplicate_qa(raw_qa)

    if len(deduped) > TARGET_FINAL_QA:
        deduped = deduped[:TARGET_FINAL_QA]

    save_json(DEDUP_QA_PATH, deduped)
    print(f"중복 제거 후 최종 Q/A 수: {len(deduped)}")

    print("[6/6] 파인튜닝 JSONL 저장")
    finetune_rows = [to_finetune_messages(item) for item in deduped]
    save_jsonl(FINETUNE_JSONL_PATH, finetune_rows)

    print("\n완료")
    print(f"- 정규화 문서: {DOCS_JSON_PATH}")
    print(f"- 청크 저장: {CHUNKS_JSON_PATH}")
    print(f"- 원본 Q/A: {RAW_QA_PATH}")
    print(f"- 중복 제거 Q/A: {DEDUP_QA_PATH}")
    print(f"- 파인튜닝 JSONL: {FINETUNE_JSONL_PATH}")

    if len(deduped) < TARGET_FINAL_QA:
        print(
            f"\n[주의] 목표 {TARGET_FINAL_QA}개보다 적은 {len(deduped)}개만 확보되었습니다. "
            "GENERATION_CONFIG를 늘리거나 MAX_PASSES를 올리세요."
        )


if __name__ == "__main__":
    main()

[1/6] 문서 로드
전체 문서 수: 140
[2/6] 문서 청킹
전체 청크 수: 255

[청크 샘플 20개]
pirelli_f1_tires__chunk_1 | tires | Pirelli F1 Tires
pirelli_f1_tires__chunk_2 | tires | Pirelli F1 Tires
pirelli_f1_tires__chunk_3 | tires | Pirelli F1 Tires
pirelli_f1_tires__chunk_4 | tires | Pirelli F1 Tires
circuit_adelaide__chunk_1 | circuit | Adelaide Street Circuit
circuit_ain-diab__chunk_1 | circuit | Ain Diab
circuit_aintree__chunk_1 | circuit | Aintree
circuit_albert_park__chunk_1 | circuit | Albert Park Grand Prix Circuit
circuit_americas__chunk_1 | circuit | Circuit of the Americas
circuit_anderstorp__chunk_1 | circuit | Scandinavian Raceway
circuit_avus__chunk_1 | circuit | AVUS
circuit_bahrain__chunk_1 | circuit | Bahrain International Circuit
circuit_baku__chunk_1 | circuit | Baku City Circuit
circuit_boavista__chunk_1 | circuit | Circuito da Boavista
circuit_brands_hatch__chunk_1 | circuit | Brands Hatch
circuit_bremgarten__chunk_1 | circuit | Circuit Bremgarten
circuit_buddh__chunk_1 | circuit | Buddh Inte